# Task 0 — Custom CNN from Scratch on GTSRB
**Name:** MD Ayaan   **Roll No:** 25155053 

---

ok so in this task i have to design my own CNN from scratch and train it on the german traffic sign dataset (GTSRB).
no torchvision.models, no pretrained stuff. just basic pytorch layers.

i watched the andrew ng CNN lecture on youtube before starting this (week 1-2 of the deeplearning.ai course) — he explains how conv layers detect edges first and then build up to more complex features. that gave me the idea for the 3-block structure i used here.



GTSRB = 43 classes of german traffic signs. classification task.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
from sklearn.metrics import confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using:', device)

NUM_CLASSES = 43

## getting the dataset from kaggle

used the kaggle API to download — copy the import code directly from the dataset page

In [ ]:
# kaggle dataset import
# run this cell first if you haven't downloaded the data yet

# !pip install kaggle
# import kaggle

# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi()
# api.authenticate()
# api.dataset_download_files('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign', path='./data', unzip=True)

# i already downloaded it so commenting this out
# data should be at ./data/GTSRB/Train/

## Dataset Loading

Train folder has subfolders for each class (0 to 42). each subfolder has the images for that class.

i'm using 32x32 because:
1. traffic signs are recognizable even at low res — the shape and color is what matters
2. after 3 maxpools the spatial goes 32→16→8→4 which is exactly enough depth
3. training is way faster lol, 224x224 would take forever on my laptop

andrew ng also mentioned in his lecture that small inputs are fine when features are not detail-heavy

In [ ]:
DATA_PATH = './data/GTSRB'
TRAIN_PATH = os.path.join(DATA_PATH, 'Train')

IMG_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.2),
    transforms.ToTensor(),
    # these mean/std values are specific to gtsrb, i looked them up
    # normalizing helps the optimizer converge faster
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],
                         std=[0.2672, 0.2564, 0.2629])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3337, 0.3064, 0.3171],
                         std=[0.2672, 0.2564, 0.2629])
])

In [ ]:
class GTSRBDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# collect all paths and labels
all_samples = []
for label_idx, class_name in enumerate(sorted(os.listdir(TRAIN_PATH))):
    class_dir = os.path.join(TRAIN_PATH, class_name)
    if not os.path.isdir(class_dir):
        continue
    for fname in os.listdir(class_dir):
        if fname.endswith('.png') or fname.endswith('.ppm'):
            all_samples.append((os.path.join(class_dir, fname), label_idx))

print('total samples:', len(all_samples))

np.random.seed(42)
np.random.shuffle(all_samples)
split = int(0.8 * len(all_samples))
train_samples = all_samples[:split]
val_samples = all_samples[split:]

train_dataset = GTSRBDataset(train_samples, train_transform)
val_dataset = GTSRBDataset(val_samples, val_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

print('train:', len(train_dataset), '| val:', len(val_dataset))

In [ ]:
# quick look at what the images actually look like
imgs, labels = next(iter(train_loader))
print('batch shape:', imgs.shape)
print('label shape:', labels.shape)
print('label range:', labels.min().item(), '-', labels.max().item())

In [ ]:
mean = np.array([0.3337, 0.3064, 0.3171])
std  = np.array([0.2672, 0.2564, 0.2629])

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()
for i in range(10):
    img = imgs[i].permute(1,2,0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    axes[i].imshow(img)
    axes[i].set_title(f'class {labels[i].item()}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('sample training images')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## My CNN Architecture

spent a while thinking about this before starting to code. here's my reasoning for each choice:

**input 32x32:**
sign shape and color is enough to distinguish them, no need for 224x224. also 3 maxpools fit perfectly: 32→16→8→4

**3 conv blocks:**
from the andrew ng CNN lectures — early layers learn edges and colors, middle layers learn shapes, final layers learn full patterns. 3 blocks is the minimum to get that hierarchy. i tried 2 blocks first but val acc was stuck around 75%.

**3x3 kernels:**
two 3x3 convs in a row have the same receptive field as one 5x5 but with fewer params and an extra non-linearity (i think the VGG paper proved this, though i used AI to double check the math here lol)

**BatchNorm:**
without it my loss was really noisy in early epochs. batchnorm normalizes the activations inside the batch which makes gradients more stable. also acts as a weak regularizer.

**MaxPool 2x2:**
halves the spatial size. also gives slight translation invariance which is good for signs that might not be perfectly centered.

**channels 32→64→128:**
standard pattern — as spatial size halves, double the channels to roughly preserve the total information capacity.

**dropout 0.4 + 0.3:**
first version of the model had a big gap between train acc and val acc. added dropout and it helped a lot. 0.4 before first FC, 0.3 before second (less aggressive since we're closer to the output)

**no skip connections:**
this is the main weakness vs ResNet. in task 2 i add those.

In [ ]:
class MyCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(MyCNN, self).__init__()

        # block 1: 3->32 channels | 32x32 -> 16x16
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # padding=1 keeps spatial size same
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # block 2: 32->64 channels | 16x16 -> 8x8
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # block 3: 64->128 channels | 8x8 -> 4x4
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        # after 3 maxpools on 32x32: 32->16->8->4
        # flat = 128 * 4 * 4 = 2048
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
            # no softmax — CrossEntropyLoss does that internally
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

In [ ]:
# make sure output shape is right before training
model = MyCNN(num_classes=43).to(device)

dummy = torch.zeros(1, 3, 32, 32).to(device)
with torch.no_grad():
    out = model(dummy)
print('output shape:', out.shape)  # should be [1, 43]

total = sum(p.numel() for p in model.parameters())
print(f'total params: {total:,}')

## Training

using Adam with lr=1e-3 and StepLR scheduler (halve lr every 10 epochs).

i tried SGD first but it was really slow to converge. adam just works better out of the box for small experiments like this.

crossentropy for classification (standard choice, nothing fancy here)

In [ ]:
model = MyCNN(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()

# adam because SGD was too slow in my first attempt
# optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)  # this was slow

optimizer = optim.Adam(model.parameters(), lr=1e-3)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_losses, val_losses = [], []

train_accs, val_accs = [], []

best_val_acc = 0.0
NUM_EPOCHS = 30




for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    
    train_acc = correct / total

    
    
    
    
    
    
    
    model.eval()
    val_loss_sum = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item()
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / len(val_loader)
    val_acc = val_correct / val_total

    
    
    
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'mycnn_best.pth')

    if (epoch+1) % 5 == 0:
        print(f'epoch {epoch+1}/{NUM_EPOCHS} | loss: {train_loss:.3f} | acc: {train_acc:.3f} | val_loss: {val_loss:.3f} | val_acc: {val_acc:.3f}')

print(f'best val acc: {best_val_acc:.4f}')

## Visualizations

plotting loss/acc curves, confusion matrix, sample predictions and per-class accuracy.
i honestly got help from AI for most of the matplotlib stuff here — i know what the plots mean but the syntax for seaborn heatmaps and subplot formatting i looked up / used AI for. still learning that part.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(train_losses, label='train', color='royalblue')
ax1.plot(val_losses, label='val', color='tomato')
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss')
ax1.set_title('loss curves')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='train', color='royalblue')
ax2.plot(val_accs, label='val', color='tomato')
ax2.set_xlabel('epoch')
ax2.set_ylabel('accuracy')
ax2.set_title('accuracy curves')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('mycnn_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# confusion matrix — used AI help for the seaborn formatting here
model.load_state_dict(torch.load('mycnn_best.pth', map_location=device))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in val_loader:
        outputs = model(imgs.to(device))
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(15, 13))
sns.heatmap(cm, cmap='Blues', fmt='d',
            xticklabels=range(43), yticklabels=range(43))
plt.title('confusion matrix - custom cnn on gtsrb')
plt.ylabel('true label')
plt.xlabel('predicted label')
plt.tight_layout()
plt.savefig('mycnn_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

final_acc = np.diag(cm).sum() / cm.sum()
print(f'final accuracy: {final_acc:.4f}')

In [ ]:
class_names = [
    'Speed 20', 'Speed 30', 'Speed 50', 'Speed 60', 'Speed 70',
    'Speed 80', 'End 80', 'Speed 100', 'Speed 120', 'No Passing',
    'No Passing >3.5t', 'Priority', 'Priority Road', 'Yield', 'Stop',
    'No Vehicles', 'No >3.5t', 'No Entry', 'General Caution', 'Dangerous Left',
    'Dangerous Right', 'Double Curve', 'Bumpy Road', 'Slippery Road', 'Road Narrows',
    'Road Work', 'Traffic Signals', 'Pedestrians', 'Children', 'Bicycles',
    'Ice/Snow', 'Wild Animals', 'End Restrictions', 'Turn Right', 'Turn Left',
    'Go Straight', 'Straight or Right', 'Straight or Left', 'Keep Right', 'Keep Left',
    'Roundabout', 'End No Passing', 'End No Passing >3.5t'
]

mean = np.array([0.3337, 0.3064, 0.3171])
std  = np.array([0.2672, 0.2564, 0.2629])

sample_imgs, sample_labels = next(iter(val_loader))
with torch.no_grad():
    sample_preds = model(sample_imgs.to(device)).argmax(dim=1).cpu().numpy()
sample_labels = sample_labels.numpy()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i in range(10):
    img = sample_imgs[i].permute(1,2,0).numpy()
    img = np.clip(img * std + mean, 0, 1)
    correct = sample_labels[i] == sample_preds[i]
    color = 'green' if correct else 'red'
    axes[i].imshow(img)
    axes[i].set_title(
        f'true: {class_names[sample_labels[i]]}\npred: {class_names[sample_preds[i]]}',
        color=color, fontsize=7
    )
    axes[i].axis('off')

plt.suptitle('predictions (green=correct, red=wrong)')
plt.tight_layout()
plt.savefig('mycnn_sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# per class accuracy — AI helped with the bar coloring syntax
per_class_acc = np.diag(cm) / cm.sum(axis=1)

plt.figure(figsize=(14, 5))
bars = plt.bar(range(43), per_class_acc, color='royalblue')

worst5 = np.argsort(per_class_acc)[:5]
for idx in worst5:
    bars[idx].set_color('tomato')

plt.axhline(y=per_class_acc.mean(), color='black', linestyle='--', label=f'mean: {per_class_acc.mean():.2f}')
plt.xlabel('class')
plt.ylabel('accuracy')
plt.title('per-class accuracy (red = worst 5)')
plt.legend()
plt.grid(axis='y')
plt.tight_layout()
plt.savefig('mycnn_per_class_acc.png', dpi=100, bbox_inches='tight')
plt.show()

print('worst 5 classes:')
for idx in worst5:
    print(f'  class {idx} ({class_names[idx]}): {per_class_acc[idx]:.3f}')

## Summary

quick recap of decisions:

- **32x32**: enough resolution for signs, fast to train, 3 maxpools fit cleanly
- **3 blocks**: low→mid→high level features, going deeper makes feature maps too tiny
- **3x3 kernels**: two 3x3 = receptive field of 5x5 but cheaper + extra relu
- **batchnorm**: stabilizes training, loss was really noisy without it
- **maxpool**: halves spatial dims, slight translation invariance
- **32→64→128**: spatial halves → double channels to keep info capacity
- **dropout 0.4/0.3**: fixed the train/val gap i was seeing
- **adam**: converges faster than SGD for this scale

main limitation: no skip connections, so this model would struggle to go deeper (vanishing gradients). that's exactly what ResNet solves in task 2.